# 초구 예측 — model_13 (3단계: 위치 1단계 보완 = 분산 따로 당기기)

- 위치 1단계 보완 (README 5장 3번): 시즌 초 배수 (model_11), 모든 카운트 위치 성향 (model_12) 채택
- 0장: model_12 위에서 남은 두 후보 다시 잼
  - 0-1 투수 전체 성향: 교차 −0.07 ×1000 -> 남는 몫 없음
  - 0-2 분산 따로 당기기: 다른 카운트 퍼짐을 쓰면 교차 +1.58 (95% −1.10 ~ +4.45) -> 방향은 +, 0과 구별 안 됨
- 모델: model_12에서 **위치 분산만 따로** (사용자 결정 (가): 만들어서 2020까지 확인)
  - 평균은 model_12 그대로, 분산·공분산 = 투수×구종×손 다른 카운트 퍼짐(투수 안 대비 비율 × 초구 투수 안 분산)을 초구 투수 안 분산 쪽으로 k_var만큼 당김
  - k_hand·분산 파라미터를 함께 2019로 다시 고르고, 평균 쪽 파라미터는 주변 작은 격자로 확인
  - 구종(model_09)은 그대로 -> 실전 변화 = 순수한 위치 개선 효과
- 비교 대상: model_12

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys
if not os.path.exists("fp_common.py"):  # model/ 폴더에서 열면 한 단계 위(데이터·fp_common·results 있는 곳)로
    os.chdir("..")
sys.path.insert(0, os.getcwd())  # fp_common을 찾도록
import numpy as np
import pandas as pd
import fp_common as fc

MODEL = "model_13"

m, names = fc.load_data()
fp, y_type, y_zone, val, test = fc.first_pitches(m)
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]
print(f"초구 {len(fp):,}개 (검증 {val.sum():,} / 평가 {test.sum():,})")
import lightgbm as lgb
from scipy.special import log_softmax
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

초구 50,709개 (검증 13,458 / 평가 12,615)


## 0. 모델 전 확인

### 0-1. model_12 위 투수 전체 성향

- 투수 전체 성향 δ_A = 그 투수(×손)가 구종과 상관없이 기준보다 치우친 만큼 (전날까지 잔차 반감기 가중 합 / (개수 + k_p))
- 넣는 방식 (model_12 0-1-3과 같음): 모든 카운트 δ를 0 대신 δ_A 쪽으로 당김 -> δ = (S_B + k_b·δ_A) / (N_B + k_b)
  - 그 구종 다른 카운트 기록이 적을수록 투수 전체 성향을 따름, 많으면 그 구종 자기 성향
- 재료 두 가지
  - **초구**: 초구 위치 − 그 공 기준점 (model_12 0-1-1과 같음)
  - **모든 카운트**: 0-0 아닌 공 위치 − 카운트×구종×손 누적 평균 (model_12 모든 카운트 δ와 같은 잔차) -> 구종 무관으로 묶으면 표본이 훨씬 큼
- 파라미터: 재료, 묶음(투수 / 투수×손), δ_A 반감기 hl_p, k_p, k_b (k_hand 19.9·위치 반감 42.1·δ 반감 12.5·시즌 초 배수 0.707은 model_12 값)
- 기준: 2019 실제 구종 기준 24칸 log loss, 이득 ×1000 = model_12 − 후보 (2019 1등 / 경기 5겹 교차), 2020 안 봄

In [2]:
DIAG = "model_12"
pp16 = fc.load_results([DIAG])[DIAG]  # 재현 확인용
HL16, KT16, KH16, SF16 = 42.1, 1000.0, 19.9, 0.707  # model_12 위치: 반감기, k_type, k_hand, 시즌 초 배수
AC_REF, AC_HL, AC_KB = None, 12.5, 17.3  # model_12 모든 카운트: 잔차 기준 반감 (None = 누적), δ 반감, k_b
REF_HL = 173.0  # 기준점 반감기
ZM = 32  # 24칸 적분 점 수

L_COLS = MOM + ZONES  # 당길 열: 위치 모멘트 5개 + 24칸 원-핫
K1, K2 = ["pitcher_code", "pitch_type"], ["pitcher_code", "pitch_type", "same_side"]  # 투수×구종 / 투수×구종×손
KR = ["pitch_type", "same_side"]  # 기준점 그룹

def loc_league(q, year, ref="all", ref_hl=None):  # 기준점 (n, 29) (model_12과 같음)
    past = fp[fp["GYEAR"] < year]
    overall = past[L_COLS].mean().to_numpy()
    if ref == "decay":
        S, N = fc.lookup(fc.prior_table(fp, KR, L_COLS, ref_hl), q, KR, L_COLS)
        return np.where(N > 0, S / np.maximum(N, 1e-12), overall)
    if ref == "last":
        past = fp[fp["GYEAR"] == year - 1]
    lg = past.groupby(KR)[L_COLS].mean()
    pr = lg.reindex(pd.MultiIndex.from_frame(q[KR]))
    return pr.fillna(pd.Series(overall, index=L_COLS)).to_numpy()

def loc_sn(q, half_life, season_f, cache=True):  # 투수×구종, 투수×구종×손 전날까지 초구 위치 S·N (model_12과 같음)
    return [fc.lookup(fc.prior_table(fp, keys, L_COLS, half_life, season_f=season_f, cache=cache), q, keys, L_COLS, season_f=season_f)
            for keys in (K1, K2)]

def shift_mean(M, dx, dz):  # 모멘트의 평균만 옮김, 분산·공분산 그대로 (model_12과 같음)
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    nx, nz = mx + dx, mz + dz
    M[:, 2] += nx ** 2 - mx ** 2
    M[:, 3] += nz ** 2 - mz ** 2
    M[:, 4] += nx * nz - mx * mz
    M[:, 0], M[:, 1] = nx, nz
    return M

MO = m[~m["label_doubt"] & ~(m["ball_before"].eq(0) & m["strike_before"].eq(0))].copy()  # 0-0 아닌 공 (구종 의심 제외)
MO["x"], MO["zn"] = MO["plate_x_in_CM"], MO["z_norm"]
MO["cnt"] = MO["ball_before"].astype(int).astype(str) + "-" + MO["strike_before"].astype(int).astype(str)
CK = ["cnt", "pitch_type", "same_side"]
S, N = fc.lookup(fc.prior_table(MO, CK, ["x", "zn"], AC_REF, cache=False), MO, CK, ["x", "zn"])  # 카운트×구종×손 누적 평균 (model_12 잔차 기준)
ref_c = np.where(N > 0, S / np.maximum(N, 1e-12), np.nan)
MO_RES = MO[K2 + ["game_date", "gameID", "pit_no"]].assign(rx=MO["x"].to_numpy() - ref_c[:, 0], rz=MO["zn"].to_numpy() - ref_c[:, 1])  # 다른 카운트 잔차

i19 = np.arange(len(q_val))
yz19 = q_val["zone"].map(fc.ZONE_IDX).to_numpy()
gid19 = q_val["gameID"].to_numpy()
REF19 = loc_league(q_val, fc.VAL_YEAR, "decay", REF_HL)
(S1_, N1_), (S2_, N2_) = loc_sn(q_val, HL16, SF16)
SB, NB = fc.lookup(fc.prior_table(MO_RES, K2, ["rx", "rz"], AC_HL, season_f=SF16, cache=False), q_val, K2, ["rx", "rz"], season_f=SF16)  # 모든 카운트 δ 재료

def ll_with(dA=None, k_b=AC_KB):  # 모든 카운트 δ를 δ_A 쪽으로 당긴 위치의 2019 공별 24칸 log loss (dA None = model_12)
    d = (SB + k_b * (0.0 if dA is None else dA)) / (NB + k_b)
    M = fc.shrink(S2_, N2_, fc.shrink(S1_, N1_, shift_mean(REF19, d[:, 0], d[:, 1]), KT16), KH16)
    P = fc.zone_from_gauss_cov(fc.gauss(M, q_val), q_val, ZM)
    return -np.log(np.clip(P[i19, yz19], 1e-12, None))

LL16 = ll_with()
assert np.allclose(LL16, fc.align(pp16, fp, fc.VAL_YEAR)["ll_zone_true"].to_numpy()), "model_12 위치 재현 실패"
print(f"model_12 2019 24칸 log loss (실제 구종) {LL16.mean():.5f} (재현 확인)")

# 투수 전체 성향 재료: 초구 잔차 (초구 위치 − 그 공 기준점)
REF_ALL = np.full((len(fp), 2), np.nan)
for yr in range(int(fp["GYEAR"].min()), fc.TEST_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    REF_ALL[k] = loc_league(fp[k], yr, "decay", REF_HL)[:, :2]
FP_RES = fp[K2 + ["game_date", "gameID", "pit_no"]].assign(rx=fp["x"].to_numpy() - REF_ALL[:, 0], rz=fp["zn"].to_numpy() - REF_ALL[:, 1])
SRC = {"초구": FP_RES, "모든 카운트": MO_RES}
PKEYS = {"투수": ["pitcher_code"], "투수×손": ["pitcher_code", "same_side"]}

# 채점 도구 (model_12 0장과 같음)
games = np.unique(gid19)
FOLD = pd.Series(np.random.default_rng(0).permutation(len(games)) % 5, index=games).reindex(gid19).to_numpy()  # 경기 단위 5겹

def game_ci(d, B=1000):  # 경기 단위 부트스트랩 95% 구간
    s = pd.DataFrame({"g": gid19, "d": d}).groupby("g")["d"].agg(["sum", "count"])
    i = np.random.default_rng(0).integers(0, len(s), (B, len(s)))
    return np.percentile(s["sum"].to_numpy()[i].sum(1) / s["count"].to_numpy()[i].sum(1), [2.5, 97.5])

def cv_ll(LL):  # 5겹 교차: 나머지 4겹 1등 조합으로 그 겹 채점
    out = np.empty(LL.shape[1])
    for f in range(5):
        b = LL[:, FOLD != f].mean(1).argmin()
        out[FOLD == f] = LL[b, FOLD == f]
    return out

_DA = {}
def delta_a(src, keys, hl_p):  # 2019 공마다 투수(×손) 잔차 합 S·개수 N (n, 2), 전날까지·시즌 초 배수
    if (src, keys, hl_p) not in _DA:
        ks = PKEYS[keys]
        _DA[(src, keys, hl_p)] = fc.lookup(fc.prior_table(SRC[src], ks, ["rx", "rz"], hl_p, season_f=SF16, cache=False), q_val, ks, ["rx", "rz"], season_f=SF16)
    return _DA[(src, keys, hl_p)]

def ll_a(src, keys, hl_p, k_p, k_b):
    S, N = delta_a(src, keys, hl_p)
    return ll_with(S / (N + k_p), k_b)  # δ_A = 투수 치우침 (0 쪽으로 k_p만큼)

RES = {}
def run_grid(name, grid):  # 모든 조합 채점 -> 1등 순 표, 범위 끝 표시
    combos = pd.MultiIndex.from_product(grid.values(), names=list(grid)).to_frame(index=False)
    recs = combos.astype(object).to_dict("records")
    LL = np.stack([ll_a(**r) for r in recs])
    tab = combos.assign(logloss=LL.mean(1), **{"이득 ×1000": (LL16.mean() - LL.mean(1)) * 1000})
    RES[name] = (tab, LL)
    top = tab.sort_values("logloss")
    edge = [k for k in grid if len(grid[k]) > 1 and top.iloc[0][k] in (grid[k][0], grid[k][-1])]
    print(f"{name}: 조합 {len(tab)}개, 1등 {top.iloc[0][list(grid)].to_dict()}, 범위 끝: {edge or '없음'}")
    display(top.head(6).round(5))
    for k in grid:  # 파라미터별 값마다 1등 (민감도)
        if len(grid[k]) > 1:
            print("  ", k, tab.groupby(k)["logloss"].min().round(5).to_dict())
    return top

GRID_P = {"src": ["초구"], "keys": ["투수", "투수×손"], "hl_p": [100.0, 300.0, 1000.0], "k_p": [30.0, 100.0, 300.0, 1000.0], "k_b": [10.0, 17.3, 30.0]}
GRID_M = {"src": ["모든 카운트"], "keys": ["투수", "투수×손"], "hl_p": [300.0, 1000.0, 3000.0], "k_p": [30.0, 100.0, 300.0, 1000.0], "k_b": [10.0, 17.3, 30.0]}
TOP_P = run_grid("재료 초구", GRID_P)
TOP_M = run_grid("재료 모든 카운트", GRID_M)

model_12 2019 24칸 log loss (실제 구종) 2.89006 (재현 확인)
재료 초구: 조합 72개, 1등 {'src': '초구', 'keys': '투수', 'hl_p': 300.0, 'k_p': 300.0, 'k_b': 17.3}, 범위 끝: ['keys']


,src,keys,hl_p,k_p,k_b,logloss,이득 ×1000
19,초구,투수,300.0,300.0,17.3,2.88996,0.09770
31,초구,투수,1000.0,300.0,17.3,2.88996,0.09334
34,초구,투수,1000.0,1000.0,17.3,2.88997,0.08825
7,초구,투수,100.0,300.0,17.3,2.88998,0.07431
22,초구,투수,300.0,1000.0,17.3,2.88998,0.07220
16,초구,투수,300.0,100.0,17.3,2.89001,0.04901


   keys {'투수': 2.88996, '투수×손': 2.89006}
   hl_p {100.0: 2.88998, 300.0: 2.88996, 1000.0: 2.88996}
   k_p {30.0: 2.8901, 100.0: 2.89001, 300.0: 2.88996, 1000.0: 2.88997}
   k_b {10.0: 2.89034, 17.3: 2.88996, 30.0: 2.89004}
재료 모든 카운트: 조합 72개, 1등 {'src': '모든 카운트', 'keys': '투수', 'hl_p': 300.0, 'k_p': 300.0, 'k_b': 30.0}, 범위 끝: ['keys', 'hl_p', 'k_b']


,src,keys,hl_p,k_p,k_b,logloss,이득 ×1000
8,모든 카운트,투수,300.0,300.0,30.0,2.88992,0.13853
22,모든 카운트,투수,1000.0,1000.0,17.3,2.88994,0.11683
7,모든 카운트,투수,300.0,300.0,17.3,2.88994,0.11547
10,모든 카운트,투수,300.0,1000.0,17.3,2.88995,0.10401
34,모든 카운트,투수,3000.0,1000.0,17.3,2.88997,0.08661
23,모든 카운트,투수,1000.0,1000.0,30.0,2.88998,0.07999


   keys {'투수': 2.88992, '투수×손': 2.89002}
   hl_p {300.0: 2.88992, 1000.0: 2.88994, 3000.0: 2.88997}
   k_p {30.0: 2.89012, 100.0: 2.88998, 300.0: 2.88992, 1000.0: 2.88994}
   k_b {10.0: 2.89036, 17.3: 2.88994, 30.0: 2.88992}


In [3]:
# 범위 끝 확인: 재료 모든 카운트 1등의 hl_p(300, 아래쪽 끝)·k_b(30, 위쪽 끝)를 넓힘 (묶음 투수 / 투수×손은 범위가 아니라 두 가지 중 고르기)
GRID_M2 = {"src": ["모든 카운트"], "keys": ["투수"], "hl_p": [100.0, 150.0, 300.0], "k_p": [100.0, 300.0, 1000.0], "k_b": [30.0, 54.8]}
TOP_M2 = run_grid("재료 모든 카운트 (넓힘)", GRID_M2)


재료 모든 카운트 (넓힘): 조합 18개, 1등 {'src': '모든 카운트', 'keys': '투수', 'hl_p': 300.0, 'k_p': 300.0, 'k_b': 30.0}, 범위 끝: ['hl_p', 'k_b']


,src,keys,hl_p,k_p,k_b,logloss,이득 ×1000
14,모든 카운트,투수,300.0,300.0,30.0,2.88992,0.13853
8,모든 카운트,투수,150.0,300.0,30.0,2.88997,0.08478
12,모든 카운트,투수,300.0,100.0,30.0,2.88998,0.07951
16,모든 카운트,투수,300.0,1000.0,30.0,2.89001,0.04537
6,모든 카운트,투수,150.0,100.0,30.0,2.89001,0.04333
2,모든 카운트,투수,100.0,300.0,30.0,2.89002,0.03133


   hl_p {100.0: 2.89002, 150.0: 2.88997, 300.0: 2.88992}
   k_p {100.0: 2.88998, 300.0: 2.88992, 1000.0: 2.89001}
   k_b {30.0: 2.88992, 54.8: 2.89024}


#### 0-1 요약

- 이득 ×1000 = model_12 − 후보 (양수 = 좋아짐), 교차 = 고르는 몫을 뺀 크기, "둘 중 고름" = 재료까지 교차 안에서 고름
- 비교: k_b만 다시 고른 경우 (투수 전체 성향 없이) -> k_b를 다시 고른 몫과 구분

In [4]:
LL_KB = np.stack([ll_with(None, kb) for kb in [10.0, 17.3, 30.0]])  # 투수 전체 성향 없이 k_b만
rows = {}
for nm, LL in [("k_b만 다시 고름 (대조)", LL_KB), ("재료 초구", RES["재료 초구"][1]), ("재료 모든 카운트", RES["재료 모든 카운트"][1]),
               ("재료 모든 카운트 (넓힘 포함)", np.vstack([RES["재료 모든 카운트"][1], RES["재료 모든 카운트 (넓힘)"][1]])),
               ("둘 중 고름", np.vstack([RES["재료 초구"][1], RES["재료 모든 카운트"][1], RES["재료 모든 카운트 (넓힘)"][1]]))]:
    b = LL.mean(1).argmin()
    d1, dc = LL16 - LL[b], LL16 - cv_ll(LL)
    (l1, h1), (lc, hc) = game_ci(d1), game_ci(dc)
    rows[nm] = {"2019 log loss": LL[b].mean(), "1등 이득": d1.mean() * 1000, "1등 95%": f"{l1 * 1000:+.2f} ~ {h1 * 1000:+.2f}",
                "교차 이득": dc.mean() * 1000, "교차 95%": f"{lc * 1000:+.2f} ~ {hc * 1000:+.2f}"}
display(pd.DataFrame(rows).T)

,2019 log loss,1등 이득,1등 95%,교차 이득,교차 95%
k_b만 다시 고름 (대조),2.890055,0.0,+0.00 ~ +0.00,-0.207771,-0.40 ~ -0.05
재료 초구,2.889958,0.097705,-0.19 ~ +0.39,-0.261925,-0.64 ~ +0.11
재료 모든 카운트,2.889917,0.138532,-0.34 ~ +0.57,-0.068196,-0.47 ~ +0.30
재료 모든 카운트 (넓힘 포함),2.889917,0.138532,-0.34 ~ +0.57,-0.068196,-0.47 ~ +0.30
둘 중 고름,2.889917,0.138532,-0.34 ~ +0.57,-0.068196,-0.47 ~ +0.30


#### 🔍 0-1 결과: model_12 위 투수 전체 성향은 남는 몫 없음
| | 2019 1등 이득 ×1000 (95%) | 교차 이득 ×1000 (95%) |
| --- | --- | --- |
| k_b만 다시 고름 (대조) | 0 | −0.21 (−0.40 ~ −0.05) |
| 재료 초구 | +0.10 (−0.19 ~ +0.39) | −0.26 (−0.64 ~ +0.11) |
| 재료 모든 카운트 | +0.14 (−0.34 ~ +0.57) | −0.07 (−0.47 ~ +0.30) |

- 2019 1등도 0과 구별 안 되고, 교차(고르는 몫을 뺀 크기)는 음수 -> 격자에서 고르는 잡음 수준
- 재료를 모든 카운트로 바꿔 표본을 늘려도 같음, 묶음은 투수가 투수×손보다 조금 나음
- 범위 끝 확인: 모든 카운트 재료 hl_p 100·150은 300보다, k_b 54.8은 30보다 나쁨 -> 1등은 안쪽
- model_12 0장 단독 교차 +0.26도 0과 구별 안 됐음 (4-25) -> 투수 구종 무관 치우침은 구종별 기록(초구 + 다른 카운트)이 이미 담고 있음
- -> 적용 안 함 후보


### 0-2. model_12 위 분산 따로 당기기 (사용자 요청)

- 지금: 위치 모멘트 x²·zn²·x·zn도 평균과 같이 기준점 쪽으로 당김 -> 퍼짐 목표에 투수끼리 평균 차이가 섞일 수 있음
- 바꿈: 평균은 model_12 그대로, 분산·공분산만 따로
  - 분산 = (N·자기 분산 + k_var·투수 안 분산) / (N + k_var), unc 켜면 × (1 + 1/(N + k_hand)) (평균을 덜 알수록 넓힘)
  - 투수 안 분산 V0 = 구종×손별 지난 시즌까지 투수×구종×손 초구 분산의 (공 수 − 1) 가중 평균
- 자기 분산 재료 두 가지 (투수×구종×손, 전날까지 반감기 hl_var, 시즌 초 배수 0.707)
  - **초구**: 초구 x·zn의 분산·공분산 (model_12 0-1-4와 같음)
  - **모든 카운트**: 0-0 아닌 공의 카운트 효과를 뺀 잔차 분산 -> 카운트마다 퍼짐 자체가 다를 수 있어 **비율**로 씀: 자기 잔차 분산 / 구종×손별 투수 안 잔차 분산 × V0 (공분산은 두 SD 비율의 곱)
- 대조: k_hand만 다시 고름 (분산 따로 없이)

In [5]:
MCOLS = ["x", "zn", "xx", "zz", "xz"]  # 초구 위치 모멘트
MO_MOM = MO_RES.assign(xx=MO_RES["rx"] ** 2, zz=MO_RES["rz"] ** 2, xz=MO_RES["rx"] * MO_RES["rz"]).rename(columns={"rx": "x", "rz": "zn"})  # 다른 카운트 잔차 모멘트

def within(data, year):  # 구종×손별 투수 안 분산·공분산 (지난 시즌까지 투수×구종×손 분산의 (공 수 − 1) 가중 평균) -> 2019 공마다 (n, 3)
    past = data[data["game_date"].dt.year < year].dropna(subset=["x", "zn"])
    g = past.groupby(K2)
    w = (g.size() - 1).clip(lower=0)
    c = pd.DataFrame({"vx": g["x"].var(), "vz": g["zn"].var(),
                      "cxz": (g["xz"].mean() - g["x"].mean() * g["zn"].mean()) * g.size() / (g.size() - 1).clip(lower=1)}).fillna(0)  # 표본 공분산
    num = c.mul(w, axis=0).groupby(level=["pitch_type", "same_side"]).sum()
    v = num.div(w.groupby(level=["pitch_type", "same_side"]).sum(), axis=0)
    allv = c.mul(w, axis=0).sum() / w.sum()  # 구종×손 기록 없을 때
    return v.reindex(pd.MultiIndex.from_frame(q_val[KR])).fillna(allv).to_numpy()

V0 = within(fp, fc.VAL_YEAR)  # 초구 투수 안 분산 (n, 3)
V0_MO = within(MO_MOM, fc.VAL_YEAR)  # 다른 카운트 잔차 투수 안 분산 (n, 3)
print("투수 안 SD 평균 (좌우 cm, 높이 비율): 초구", np.sqrt(V0[:, :2]).mean(0).round(3), "/ 다른 카운트 잔차", np.sqrt(V0_MO[:, :2]).mean(0).round(3))
d_ac = (SB + AC_KB * 0.0) / (NB + AC_KB)  # model_12 모든 카운트 δ
P_AC = shift_mean(REF19, d_ac[:, 0], d_ac[:, 1])  # model_12 출발점

def mean_M(k_hand):  # model_12 모멘트 (k_hand만 바꿈)
    return fc.shrink(S2_, N2_, fc.shrink(S1_, N1_, P_AC, KT16), k_hand)

def ll_of(M):  # 모멘트 -> 2019 공별 24칸 log loss
    P = fc.zone_from_gauss_cov(fc.gauss(M, q_val), q_val, ZM)
    return -np.log(np.clip(P[i19, yz19], 1e-12, None))

assert np.allclose(ll_of(mean_M(KH16)), LL16)
SRC_V = {"초구": fp, "모든 카운트": MO_MOM}
_DV = {}
def own_var(src, hl_var):  # 2019 공마다 자기 분산·공분산 (n, 3)과 (가중) 공 수 (n,)
    if (src, hl_var) not in _DV:
        S, N = fc.lookup(fc.prior_table(SRC_V[src], K2, MCOLS, hl_var, season_f=SF16, cache=False), q_val, K2, MCOLS, season_f=SF16)
        e = S / np.maximum(N, 1e-12)
        v = np.column_stack([e[:, 2] - e[:, 0] ** 2, e[:, 3] - e[:, 1] ** 2, e[:, 4] - e[:, 0] * e[:, 1]])
        if src == "모든 카운트":  # 비율로 바꿈: 자기 / 투수 안 (다른 카운트) × 초구 투수 안
            r = np.sqrt(np.maximum(v[:, :2], 0) / V0_MO[:, :2])  # 좌우·높이 SD 비율
            v = np.column_stack([V0[:, 0] * r[:, 0] ** 2, V0[:, 1] * r[:, 1] ** 2, v[:, 2] / np.sqrt(V0_MO[:, 0] * V0_MO[:, 1]) * np.sqrt(V0[:, 0] * V0[:, 1])])
        n = N[:, 0]
        _DV[(src, hl_var)] = (np.where(n[:, None] > 0, np.nan_to_num(v), 0.0), n)
    return _DV[(src, hl_var)]

def ll_v(src, hl_var, k_var, unc, k_hand):
    M = mean_M(k_hand)
    mx, mz = M[:, 0], M[:, 1]
    v, n = own_var(src, hl_var)
    V = (n[:, None] * v + k_var * V0) / (n[:, None] + k_var)  # 투수 안 분산 쪽으로 당김
    if unc:
        V = V * (1 + 1 / (N2_[:, :1] + k_hand))
    M2 = M.copy()
    M2[:, 2], M2[:, 3], M2[:, 4] = V[:, 0] + mx ** 2, V[:, 1] + mz ** 2, V[:, 2] + mx * mz  # 분산 -> 모멘트
    return ll_of(M2)

def run_grid_v(name, grid):
    combos = pd.MultiIndex.from_product(grid.values(), names=list(grid)).to_frame(index=False)
    LL = np.stack([ll_v(**r) for r in combos.astype(object).to_dict("records")])
    tab = combos.assign(logloss=LL.mean(1), **{"이득 ×1000": (LL16.mean() - LL.mean(1)) * 1000})
    RES[name] = (tab, LL)
    top = tab.sort_values("logloss")
    edge = [k for k in grid if len(grid[k]) > 1 and not isinstance(grid[k][0], (str, bool)) and top.iloc[0][k] in (grid[k][0], grid[k][-1])]
    print(f"{name}: 조합 {len(tab)}개, 1등 {top.iloc[0][list(grid)].to_dict()}, 범위 끝: {edge or '없음'}")
    display(top.head(6).round(5))
    for k in grid:
        if len(grid[k]) > 1:
            print("  ", k, tab.groupby(k)["logloss"].min().round(5).to_dict())
    return top

GRID_VP = {"src": ["초구"], "hl_var": [42.1, 100.0, 300.0], "k_var": [3.0, 10.0, 30.0, 100.0], "unc": [False, True], "k_hand": [17.3, 19.9, 22.8]}
GRID_VM = {"src": ["모든 카운트"], "hl_var": [12.5, 50.0, 200.0], "k_var": [3.0, 10.0, 30.0, 100.0], "unc": [False, True], "k_hand": [17.3, 19.9, 22.8]}
TOP_VP = run_grid_v("분산 재료 초구", GRID_VP)
TOP_VM = run_grid_v("분산 재료 모든 카운트", GRID_VM)

투수 안 SD 평균 (좌우 cm, 높이 비율): 초구 [24.856  0.516] / 다른 카운트 잔차 [25.545  0.525]
분산 재료 초구: 조합 72개, 1등 {'src': '초구', 'hl_var': 100.0, 'k_var': 10.0, 'unc': False, 'k_hand': 22.8}, 범위 끝: ['k_hand']


,src,hl_var,k_var,unc,k_hand,logloss,이득 ×1000
32,초구,100.0,10.0,False,22.8,2.88949,0.56813
56,초구,300.0,10.0,False,22.8,2.88963,0.42239
31,초구,100.0,10.0,False,19.9,2.88974,0.31481
35,초구,100.0,10.0,True,22.8,2.88985,0.20259
55,초구,300.0,10.0,False,19.9,2.88988,0.17364
8,초구,42.1,10.0,False,22.8,2.88995,0.10602


   hl_var {42.1: 2.88995, 100.0: 2.88949, 300.0: 2.88963}
   k_var {3.0: 2.89262, 10.0: 2.88949, 30.0: 2.89045, 100.0: 2.89537}
   unc {False: 2.88949, True: 2.88985}
   k_hand {17.3: 2.89008, 19.9: 2.88974, 22.8: 2.88949}
분산 재료 모든 카운트: 조합 72개, 1등 {'src': '모든 카운트', 'hl_var': 200.0, 'k_var': 10.0, 'unc': False, 'k_hand': 22.8}, 범위 끝: ['hl_var', 'k_hand']


,src,hl_var,k_var,unc,k_hand,logloss,이득 ×1000
56,모든 카운트,200.0,10.0,False,22.8,2.88853,1.52763
55,모든 카운트,200.0,10.0,False,19.9,2.88880,1.25287
59,모든 카운트,200.0,10.0,True,22.8,2.88882,1.23933
62,모든 카운트,200.0,30.0,False,22.8,2.88907,0.98731
58,모든 카운트,200.0,10.0,True,19.9,2.88910,0.95660
54,모든 카운트,200.0,10.0,False,17.3,2.88917,0.88930


   hl_var {12.5: 2.89739, 50.0: 2.89067, 200.0: 2.88853}
   k_var {3.0: 2.89011, 10.0: 2.88853, 30.0: 2.88907, 100.0: 2.89303}
   unc {False: 2.88853, True: 2.88882}
   k_hand {17.3: 2.88917, 19.9: 2.8888, 22.8: 2.88853}


In [6]:
# 범위 끝 확인: 모든 카운트 재료 hl_var 200 (위쪽 끝)·k_hand 22.8 (위쪽 끝), 초구 재료 k_hand 22.8 (위쪽 끝) -> 넓힘
GRID_VM2 = {"src": ["모든 카운트"], "hl_var": [200.0, 400.0, 800.0, 1600.0], "k_var": [5.48, 10.0, 17.3, 30.0], "unc": [False], "k_hand": [22.8, 26.2, 30.0, 40.0]}
GRID_VP2 = {"src": ["초구"], "hl_var": [100.0], "k_var": [10.0], "unc": [False], "k_hand": [22.8, 26.2, 30.0, 40.0]}
TOP_VM2 = run_grid_v("분산 재료 모든 카운트 (넓힘)", GRID_VM2)
TOP_VP2 = run_grid_v("분산 재료 초구 (넓힘)", GRID_VP2)


분산 재료 모든 카운트 (넓힘): 조합 64개, 1등 {'src': '모든 카운트', 'hl_var': 200.0, 'k_var': 17.3, 'unc': False, 'k_hand': 30.0}, 범위 끝: ['hl_var']


,src,hl_var,k_var,unc,k_hand,logloss,이득 ×1000
10,모든 카운트,200.0,17.3,False,30.0,2.88821,1.84778
26,모든 카운트,400.0,17.3,False,30.0,2.88824,1.81915
6,모든 카운트,200.0,10.0,False,30.0,2.88824,1.81425
22,모든 카운트,400.0,10.0,False,30.0,2.88830,1.75680
9,모든 카운트,200.0,17.3,False,26.2,2.88830,1.75415
11,모든 카운트,200.0,17.3,False,40.0,2.88832,1.73072


   hl_var {200.0: 2.88821, 400.0: 2.88824, 800.0: 2.88836, 1600.0: 2.88845}
   k_var {5.48: 2.88881, 10.0: 2.88824, 17.3: 2.88821, 30.0: 2.88874}
   k_hand {22.8: 2.88849, 26.2: 2.8883, 30.0: 2.88821, 40.0: 2.88832}
분산 재료 초구 (넓힘): 조합 4개, 1등 {'src': '초구', 'hl_var': 100.0, 'k_var': 10.0, 'unc': False, 'k_hand': 30.0}, 범위 끝: 없음


,src,hl_var,k_var,unc,k_hand,logloss,이득 ×1000
2,초구,100.0,10.0,False,30.0,2.88924,0.81071
1,초구,100.0,10.0,False,26.2,2.88932,0.73816
3,초구,100.0,10.0,False,40.0,2.88941,0.64286
0,초구,100.0,10.0,False,22.8,2.88949,0.56813


   k_hand {22.8: 2.88949, 26.2: 2.88932, 30.0: 2.88924, 40.0: 2.88941}


In [7]:
LL_KH = np.stack([ll_of(mean_M(kh)) for kh in [17.3, 19.9, 22.8, 26.2, 30.0, 40.0]])  # 분산 따로 없이 k_hand만
rows = {}
for nm, LL in [("k_hand만 다시 고름 (대조)", LL_KH), ("분산 재료 초구", np.vstack([RES["분산 재료 초구"][1], RES["분산 재료 초구 (넓힘)"][1]])),
               ("분산 재료 모든 카운트", np.vstack([RES["분산 재료 모든 카운트"][1], RES["분산 재료 모든 카운트 (넓힘)"][1]])),
               ("둘 중 고름", np.vstack([RES[k][1] for k in RES if k.startswith("분산")]))]:
    b = LL.mean(1).argmin()
    d1, dc = LL16 - LL[b], LL16 - cv_ll(LL)
    (l1, h1), (lc, hc) = game_ci(d1), game_ci(dc)
    rows[nm] = {"2019 log loss": LL[b].mean(), "1등 이득": d1.mean() * 1000, "1등 95%": f"{l1 * 1000:+.2f} ~ {h1 * 1000:+.2f}",
                "교차 이득": dc.mean() * 1000, "교차 95%": f"{lc * 1000:+.2f} ~ {hc * 1000:+.2f}"}
display(pd.DataFrame(rows).T)

,2019 log loss,1등 이득,1등 95%,교차 이득,교차 95%
k_hand만 다시 고름 (대조),2.890055,0.0,+0.00 ~ +0.00,0.0,+0.00 ~ +0.00
분산 재료 초구,2.889245,0.810713,-0.69 ~ +2.31,0.810713,-0.69 ~ +2.31
분산 재료 모든 카운트,2.888208,1.847778,-0.72 ~ +4.58,1.579381,-1.10 ~ +4.45
둘 중 고름,2.888208,1.847778,-0.72 ~ +4.58,1.579381,-1.10 ~ +4.45


#### 🔍 0-2 결과: 다른 카운트 퍼짐을 쓰면 방향은 +, 95% 구간은 0을 포함
| | 1등 파라미터 | 2019 1등 이득 ×1000 (95%) | 교차 이득 ×1000 (95%) |
| --- | --- | --- | --- |
| k_hand만 다시 고름 (대조) | k_hand 19.9 그대로 | 0 | 0 |
| 분산 재료 초구 | 반감 100 · k_var 10 · k_hand 30 | +0.81 (−0.69 ~ +2.31) | +0.81 (−0.69 ~ +2.31) |
| **분산 재료 모든 카운트** | 반감 200 · k_var 17.3 · k_hand 30 | **+1.85** (−0.72 ~ +4.58) | **+1.58** (−1.10 ~ +4.45) |

- 다른 카운트 퍼짐(카운트 효과를 뺀 잔차 분산을 투수 안 분산 대비 비율로)이 초구 퍼짐보다 나음 -> 표본이 많아 투수×구종×손별 퍼짐을 더 잘 앎
- 크기(교차 +1.58)는 model_12 모든 카운트 평균(+1.47)과 비슷하지만 95% 구간이 넓어 0을 포함 -> 퍼짐을 바꾸면 공마다 log loss가 크게 흔들려 잡음이 큼
- 분산 따로를 쓰면 k_hand가 19.9 -> 30으로 커짐 (k_hand만 바꾸면 이득 0) -> 퍼짐을 따로 정하니 평균은 투수 초구 기록을 덜 따라가는 게 나음
- model_12 0장(−0.08)과 다른 이유: k_hand 범위가 22.8까지였고, 재료가 초구뿐이었음
- unc(평균을 덜 알수록 넓힘)는 두 재료 모두 끄는 게 나음
- 범위 끝 확인: hl_var 12.5 · 50 · 400 이상, k_var 5.48 · 30, k_hand 22.8 · 40 모두 1등보다 나쁨 -> 1등은 안쪽
- 4-23 진단(평균 퍼짐은 맞음, 투수별 퍼짐 차이는 해마다 안 이어짐)과 맞음: 해 단위로는 안 이어져도 최근 200구 정도의 투수별 퍼짐은 쓸모 있을 수 있음


## 모델

In [8]:
# 0장이 모델 변수 이름을 덮어쓰지 않았는지 대비: 설정 상수를 다시 정함
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]


## 1. 파라미터

- 구종은 model_09 값 고정, 위치 출발 값은 model_12 (시즌 초 배수 0.707은 고정) -> 3-1에서 다시 고른 값으로 바꿈

In [9]:
# ① 구종 출발점 B4: model_06 값 (고정) - 반감기·alpha·hand_k, 시즌 초 배수 (기준점 = 지난 시즌 전부)
B4_P = dict(half_life=35.4, alpha=7.5, hand_k=5.48, season_f=0.25)
# ① LightGBM 변수: A 이번 타석 상황 + 앞 타석 초구 결과 원-핫 (fc.FIRST_CATS)
FEAT = fc.A_COLS + ["prev_first_" + c for c in fc.FIRST_CATS]
# ② 위치: 출발 값 = model_12 -> 3-1에서 다시 고름
LOC_REF, LOC_REF_HL = "decay", 173.0  # 기준점: 분석 대상 투수 구종×손, 전날까지 반감 173구 (고정)
LOC_HL, K_TYPE, K_HAND = 42.1, 1000.0, 19.9  # 투수 기록 반감기, 투수×구종 당김 k, 투수×구종×손 당김 k (model_12)
LOC_SEASON_F = 0.707  # 시즌 초 배수 (model_11, 고정)
LOC_AC = (None, 12.5, 17.3)  # 모든 카운트 위치 성향 (잔차 기준 반감 None = 누적, δ 반감, δ 당김 k) (model_12)
LOC_VAR = None  # 분산 따로 (다른 카운트 퍼짐 반감 hl_var, 당김 k_var, unc, 투수 안 분산 당김 k0), None = 없음 (model_12)
ZONE_M = 32
# ① LightGBM: model_07 값 (고정)
LGB_P = {"num_leaves": 7, "min_data_in_leaf": 20, "lambda_l2": 1000.0}
LGB_ROUNDS = 730
RETRAIN = "month"  # 재학습 주기: 그 달 첫 경기 전날까지 데이터로 다시 학습 (3월은 4월에, 10월은 9월에 합침)
# 구종 보정 학습 해
TRAIN_FIRST = 2017

## 2. ① 구종: B4 + LightGBM 월 단위 재학습 (model_09과 같음)

### 2-1. B4 (model_06·07·09과 같음)

In [10]:
def b4_online(q, year, half_life, alpha, hand_k, season_f):  # model_06 B4 (n, 9): 지난 시즌 리그 비율 -> 투수 (alpha) -> 투수×손 (hand_k), 시즌 초 배수
    first = year == fp["GYEAR"].min()  # 첫 시즌(2017): 지난 기록 없음 -> 그 시즌 비율 (학습 데이터 출발점용)
    ref = fp.loc[fp["GYEAR"] == year if first else fp["GYEAR"] < year, CLASSES].mean().to_numpy()
    out = []
    for keys in (["pitcher_code"], ["pitcher_code", "same_side"]):
        S, N = fc.lookup(fc.prior_table(fp, keys, CLASSES, half_life, season_f=season_f), q, keys, CLASSES, season_f=season_f)
        out += [S, N[:, :1]]
    S, N, S2, N2 = out
    return fc.shrink(S2, N2, fc.shrink(S, N, ref, alpha), hand_k)

K = len(CLASSES)
P_B4 = np.full((len(fp), K), np.nan)  # 2017–2020
for yr in range(TRAIN_FIRST, fc.TEST_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    P_B4[k] = b4_online(fp[k], yr, **B4_P)
b4_19 = fc.prob_metrics(P_B4[val], y_type[val])["logloss"]
assert abs(b4_19 - 1.09367) < 1e-5, b4_19  # model_06·07 B4의 2019 log loss
print(f"2019 B4 log loss {b4_19:.5f} = model_06·07 B4 (재현 확인)")
m07 = pd.read_parquet("results/model_07/per_pitch.parquet")  # 첫 달 재현 확인용
m09 = pd.read_parquet("results/model_09/per_pitch.parquet")  # 구종 재현 확인용 (구종은 model_09 그대로)
m12 = pd.read_parquet("results/model_12/per_pitch.parquet")  # 비교 대상

2019 B4 log loss 1.09367 = model_06·07 B4 (재현 확인)


### 2-2. 변수 (model_07과 같음)

- A 이번 타석 상황 + 앞 타석 초구 결과, 미래 정보 검사 다시 통과 확인

In [11]:
X_all = pd.concat([fc.situation_features(m, fp), fc.game_flow_features(m, fp)], axis=1)[FEAT]
assert X_all.notna().all().all()
for fn in (fc.situation_features, fc.game_flow_features):
    bad = fc.check_no_future(fn, m, fp[fp["GYEAR"].ge(TRAIN_FIRST)], n=150)
    assert bad.empty, bad
display(X_all.describe().T[["mean", "min", "max"]].round(3))

미래 정보 검사: 150개 중 다른 공 0개
미래 정보 검사: 150개 중 다른 공 0개


,mean,min,max
pc,33.394,0.0,124.0
tto,1.592,1.0,5.0
starter,0.690,0.0,1.0
inning,4.548,1.0,12.0
out_cn,0.970,0.0,2.0
base1,0.359,0.0,1.0
base2,0.194,0.0,1.0
base3,0.100,0.0,1.0
batorder,4.811,1.0,9.0
sdiff,0.369,-21.0,19.0


### 2-2. 월 단위 재학습과 구종 확률 (model_09과 같음)

- 확인: 구종 확률 = model_09 결과 파일 (재현)

In [12]:
y_all = fp["pitch_type"].map(fc.TYPE_IDX).to_numpy()
INIT = np.log(np.clip(P_B4, 1e-6, None))
INIT -= INIT.mean(1, keepdims=True)  # 출발점 (공마다 상수는 softmax에 영향 없음)
BASE = dict(objective="multiclass", num_class=K, learning_rate=0.03, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
            verbose=-1, seed=0, deterministic=True, force_row_wise=True, num_threads=4)  # model_05–07과 같음
yr_all = fp["GYEAR"].to_numpy()
DATE = pd.to_datetime(fp["gameID"].str[:8]).to_numpy()
MON = np.clip(fp["gameID"].str[4:6].astype(int).to_numpy(), 4, 9)  # 재학습 묶음 달: 3월 -> 4월, 10월 -> 9월

def fit_rows(tr):  # 학습 행 tr로 LightGBM (트리 수 고정)
    return lgb.train({**BASE, **LGB_P}, lgb.Dataset(X_all.iloc[tr], y_all[tr], init_score=INIT[tr]), LGB_ROUNDS)

def type_prob(bst, idx):  # B4 출발점 + 트리 보정 -> 구종 확률 (n, 9)
    return np.exp(log_softmax(bst.predict(X_all.iloc[idx], raw_score=True) + INIT[idx], axis=1))

P_TYPE_ALL = np.full((len(fp), K), np.nan)
BST, plan = {}, []
for yr in (fc.VAL_YEAR, fc.TEST_YEAR):
    iy = np.flatnonzero(yr_all == yr)
    for mon in np.unique(MON[iy]):
        te = iy[MON[iy] == mon]
        start = DATE[te].min()  # 그 달 첫 경기일
        tr = np.flatnonzero((yr_all >= TRAIN_FIRST) & (DATE < start))
        assert DATE[tr].max() < start  # 미래 정보 없음
        BST[(yr, mon)] = bst = fit_rows(tr)
        P_TYPE_ALL[te] = type_prob(bst, te)
        plan.append({"연도": yr, "월": mon, "예측 초구": len(te), "학습 초구": len(tr), "그 시즌 학습 초구": int((yr_all[tr] == yr).sum()),
                     "학습 끝": str(pd.Timestamp(DATE[tr].max()).date()), "예측 시작": str(pd.Timestamp(start).date())})
display(pd.DataFrame(plan).set_index(["연도", "월"]))
P11 = {yr: fc.align(m07, fp, yr)[fc.P_TYPE].to_numpy() for yr in (fc.VAL_YEAR, fc.TEST_YEAR)}  # model_07 구종 확률 (fp 그 해 순서)
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:  # 첫 달 = 시즌 단위 (model_07 재현)
    k = MON[msk] == MON[msk].min()
    assert np.allclose(P_TYPE_ALL[msk][k], P11[yr][k], atol=1e-6)
print("첫 달 구종 확률 = model_07 (재현 확인)")
rows = {}
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    for nm, P in [("B4", P_B4[msk]), ("model_07", P11[yr]), (MODEL, P_TYPE_ALL[msk])]:
        r = fc.prob_metrics(P, y_type[msk])
        rows[(yr, nm)] = {"log loss": r["logloss"], "top1": r["top1"], "top2": r["top2"]}
display(pd.DataFrame(rows).T.round(4))
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    assert np.allclose(P_TYPE_ALL[msk], fc.align(m09, fp, yr)[fc.P_TYPE].to_numpy(), atol=1e-6), f"{yr} 구종 확률이 model_09과 다름"
print("구종 확률 = model_09 (재현 확인)")

예측 초구  학습 초구  그 시즌 학습 초구        학습 끝       예측 시작
연도   월                                                  
2019 4   2568  24636           0  2018-10-14  2019-03-23
     5   2403  27204        2568  2019-04-30  2019-05-01
     6   2493  29607        4971  2019-05-31  2019-06-01
     7   1791  32100        7464  2019-06-30  2019-07-02
     8   2362  33891        9255  2019-07-31  2019-08-01
     9   1841  36253       11617  2019-08-31  2019-09-01
2020 5   2264  38094           0  2019-10-01  2020-05-05
     6   2379  40358        2264  2020-05-31  2020-06-02
     7   2233  42737        4643  2020-06-30  2020-07-01
     8   1985  44970        6876  2020-07-31  2020-08-01
     9   3754  46955        8861  2020-08-30  2020-09-01

첫 달 구종 확률 = model_07 (재현 확인)


log loss    top1    top2
2019 B4          1.0937  0.5279  0.8159
     model_07    1.0621  0.5380  0.8174
     model_13    1.0613  0.5385  0.8177
2020 B4          1.0916  0.5264  0.8179
     model_07    1.0570  0.5340  0.8209
     model_13    1.0561  0.5332  0.8218

구종 확률 = model_09 (재현 확인)


## 3. ② 위치: 분산 따로 당기기

- 평균(좌우 x, 높이 비율 zn): model_12 그대로 (기준점 -> 모든 카운트 δ만큼 이동 -> 투수×구종 -> 투수×구종×손 초구 기록)
- 분산·공분산 (0-2 방식)
  - 투수 안 분산 V0: 구종×손별 지난 시즌까지 투수×구종×손 **초구** 분산의 (공 수 − 1) 가중 평균
  - 자기 퍼짐: 투수×구종×손 **다른 카운트 잔차**(카운트×구종×손 평균을 뺀 위치)의 전날까지 반감기(`hl_var`) 분산 -> 카운트마다 퍼짐이 달라 비율로: 자기 잔차 분산 / 구종×손 투수 안 잔차 분산 × V0 (공분산은 SD 비율의 곱)
  - V0·잔차 투수 안 분산은 구종×손 자유도 합이 적으면 전체 구종 값 쪽으로 k0만큼 당김 (첫 실행에서 너클 초구 3구로 V0 좌우 SD 3.9cm -> 2020 크게 나빠짐, 사용자 확인 후 고침)
  - 분산 = (N·자기 + k_var·V0) / (N + k_var), unc 켜면 × (1 + 1/(초구 N + k_hand))
  - 지금(model_12)은 x²·zn²·x·zn 평균을 평균과 같이 기준점 쪽으로 당겨 분산을 만듦
- 확인: `LOC_VAR = None`이면 model_12과 같음 (재현)

In [14]:
# 24칸 적분 함수: fc.zone_from_gauss_cov(g, q, m=32) (model_03부터)
def zone_probs(g, q, cov=True):  # 24칸 확률: cov=True 공분산 반영 (model_03) / False 독립 가정 (model_02까지)
    return fc.zone_from_gauss_cov(g, q, ZONE_M) if cov else fc.zone_from_gauss(g, q)

L_COLS = MOM + ZONES  # 당길 열: 위치 모멘트 5개(x, 높이 비율, 제곱, 곱) + 24칸 원-핫
K1, K2 = ["pitcher_code", "pitch_type"], ["pitcher_code", "pitch_type", "same_side"]  # 투수×구종 / 투수×구종×손
KR = ["pitch_type", "same_side"]  # 기준점 그룹: 분석 대상 투수 구종×손

def loc_league(q, year, ref="all", ref_hl=None):  # 기준점 (n, 29): "all" 지난 시즌 전부 / "last" 직전 시즌만 / "decay" 전날까지 반감기 ref_hl(공 수) 가중
    past = fp[fp["GYEAR"] < year]  # 지난 시즌까지
    overall = past[L_COLS].mean().to_numpy()  # 그 구종×손 기록이 없을 때 쓰는 전체 평균 (지난 시즌까지)
    if ref == "decay":
        S, N = fc.lookup(fc.prior_table(fp, KR, L_COLS, ref_hl), q, KR, L_COLS)  # 구종×손별 전날까지 가중 합계, 개수
        return np.where(N > 0, S / np.maximum(N, 1e-12), overall)  # 가중 평균, 기록 없으면 전체
    if ref == "last":
        past = fp[fp["GYEAR"] == year - 1]  # 직전 시즌만
    lg = past.groupby(KR)[L_COLS].mean()
    pr = lg.reindex(pd.MultiIndex.from_frame(q[KR]))  # 공마다 그 구종×손 행
    return pr.fillna(pd.Series(overall, index=L_COLS)).to_numpy()

def loc_sn(q, half_life, season_f, cache=True):  # 투수×구종, 투수×구종×손 전날까지 초구 위치 합계 S·개수 N 두 쌍 (시즌 초 배수 반영)
    return [fc.lookup(fc.prior_table(fp, keys, L_COLS, half_life, season_f=season_f, cache=cache), q, keys, L_COLS, season_f=season_f)
            for keys in (K1, K2)]

def shift_mean(M, dx, dz):  # 모멘트 M (n, 29)의 평균만 (dx, dz) 옮김, 분산·공분산 그대로 (24칸 원-핫 열은 정규분포에 안 쓰여 그대로)
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    nx, nz = mx + dx, mz + dz
    M[:, 2] += nx ** 2 - mx ** 2  # E[x²] = 분산 + 평균²
    M[:, 3] += nz ** 2 - mz ** 2
    M[:, 4] += nx * nz - mx * mz  # E[x·z] = 공분산 + 평균 곱
    M[:, 0], M[:, 1] = nx, nz
    return M

# 0-0 아닌 공 (구종 의심 제외): 모든 카운트 위치 성향 재료
MO = m[~m["label_doubt"] & ~(m["ball_before"].eq(0) & m["strike_before"].eq(0))].copy()
MO["x"], MO["zn"] = MO["plate_x_in_CM"], MO["z_norm"]  # 좌우 cm (타자 기준), 높이 비율 (존 아랫선 0, 윗선 1)
MO["cnt"] = MO["ball_before"].astype(int).astype(str) + "-" + MO["strike_before"].astype(int).astype(str)  # 카운트
CK = ["cnt", "pitch_type", "same_side"]  # 카운트 효과 그룹
_MO_RES = {}

def mo_resid(ref_hl):  # 0-0 아닌 공마다 위치 − 분석 대상 투수 그 카운트×구종×손 전날까지 평균 (반감 ref_hl구) -> 잔차 표 (rx 좌우 cm, rz 높이 비율)
    if ref_hl not in _MO_RES:
        S, N = fc.lookup(fc.prior_table(MO, CK, ["x", "zn"], ref_hl, cache=False), MO, CK, ["x", "zn"])
        ref = np.where(N > 0, S / np.maximum(N, 1e-12), np.nan)  # 그 카운트×구종×손 기록이 없으면 빈 값 (셈에서 빠짐)
        _MO_RES[ref_hl] = MO[K2 + ["game_date", "gameID", "pit_no"]].assign(rx=MO["x"].to_numpy() - ref[:, 0], rz=MO["zn"].to_numpy() - ref[:, 1])
    return _MO_RES[ref_hl]

def ac_sn(q, ref_hl, hl_b, season_f, cache=True):  # 공마다 투수×구종×손 다른 카운트 잔차 전날까지 합 S·개수 N (n, 2)
    d = mo_resid(ref_hl)
    return fc.lookup(fc.prior_table(d, K2, ["rx", "rz"], hl_b, season_f=season_f, cache=cache), q, K2, ["rx", "rz"], season_f=season_f)

MCOLS = ["x", "zn", "xx", "zz", "xz"]  # 위치 모멘트 (좌우, 높이 비율, 제곱, 곱)
_MO_MOM, _WT = {}, {}

def mo_mom(ref_hl):  # 다른 카운트 잔차 모멘트 표 (x = 좌우 잔차, zn = 높이 비율 잔차, 제곱·곱)
    if ref_hl not in _MO_MOM:
        d = mo_resid(ref_hl)
        _MO_MOM[ref_hl] = d.assign(xx=d["rx"] ** 2, zz=d["rz"] ** 2, xz=d["rx"] * d["rz"]).rename(columns={"rx": "x", "rz": "zn"})
    return _MO_MOM[ref_hl]

def within_tab(name, data, year):  # 구종×손별 투수 안 분산·공분산 (지난 시즌까지 투수×구종×손 분산의 (공 수 − 1) 가중 평균) 표와 전체 값
    if (name, year) not in _WT:
        past = data[data["game_date"].dt.year < year].dropna(subset=["x", "zn"])
        g = past.groupby(K2)
        n = g.size()
        w = (n - 1).clip(lower=0)
        c = pd.DataFrame({"vx": g["x"].var(), "vz": g["zn"].var(),
                          "cxz": (g["xz"].mean() - g["x"].mean() * g["zn"].mean()) * n / (n - 1).clip(lower=1)}).fillna(0)  # 표본 분산·공분산
        num = c.mul(w, axis=0).groupby(level=["pitch_type", "same_side"]).sum()
        wg = w.groupby(level=["pitch_type", "same_side"]).sum()  # 구종×손별 자유도 합 (표본 크기)
        _WT[(name, year)] = (num.div(wg, axis=0), c.mul(w, axis=0).sum() / w.sum(), wg)
    return _WT[(name, year)]

def v0_of(q, name, data, year, k0=0.0):  # 공마다 그 구종×손 투수 안 분산·공분산 (n, 3): 자유도 합이 적으면 전체 구종 값 쪽으로 k0만큼 당김
    tab, allv, wg = within_tab(name, data, year)
    idx = pd.MultiIndex.from_frame(q[KR])
    a = allv.to_numpy()[None, :]  # 전체 구종 투수 안 분산
    v = tab.reindex(idx).to_numpy()
    n = wg.reindex(idx).fillna(0).to_numpy()[:, None]  # 그 구종×손 자유도 합
    v = np.where(np.isnan(v), a, v)
    return np.where(n + k0 > 0, (n * v + k0 * a) / np.maximum(n + k0, 1e-12), a)  # 예: 너클 초구 3구(자유도 2)면 거의 전체 값

def own_var(q, year, ref_hl, hl_var, season_f, k0=0.0, cache=True):  # 공마다 다른 카운트 퍼짐 -> 초구 단위로 바꾼 자기 분산·공분산 (n, 3), (가중) 공 수 (n,), V0 (n, 3)
    data = mo_mom(ref_hl)
    S, N = fc.lookup(fc.prior_table(data, K2, MCOLS, hl_var, season_f=season_f, cache=cache), q, K2, MCOLS, season_f=season_f)
    e = S / np.maximum(N, 1e-12)
    v = np.column_stack([e[:, 2] - e[:, 0] ** 2, e[:, 3] - e[:, 1] ** 2, e[:, 4] - e[:, 0] * e[:, 1]])  # 자기 잔차 분산·공분산
    V0 = v0_of(q, "초구", fp, year, k0)
    V0m = v0_of(q, ("다른 카운트", ref_hl), data, year, k0)
    r = np.sqrt(np.maximum(v[:, :2], 0) / V0m[:, :2])  # 좌우·높이 SD 비율 (자기 / 투수 안)
    v = np.column_stack([V0[:, 0] * r[:, 0] ** 2, V0[:, 1] * r[:, 1] ** 2, v[:, 2] / np.sqrt(V0m[:, 0] * V0m[:, 1]) * np.sqrt(V0[:, 0] * V0[:, 1])])
    n = N[:, 0]
    return np.where(n[:, None] > 0, np.nan_to_num(v), 0.0), n, V0

def apply_var(M, v, n, V0, k_var, unc, N2, k_hand):  # 모멘트 M의 분산·공분산을 따로 당긴 값으로 바꿈 (평균 그대로)
    V = (n[:, None] * v + k_var * V0) / (n[:, None] + k_var)  # 투수 안 분산 쪽으로 k_var만큼
    if unc:
        V = V * (1 + 1 / (N2[:, :1] + k_hand))  # 평균을 덜 알수록 넓힘
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    M[:, 2], M[:, 3], M[:, 4] = V[:, 0] + mx ** 2, V[:, 1] + mz ** 2, V[:, 2] + mx * mz  # 분산 -> 모멘트
    return M

def loc_online(q, year, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, var=None):  # 위치 모멘트 (n, 29)
    P = loc_league(q, year, ref, ref_hl)  # 출발: 기준점
    if ac is not None:  # 모든 카운트 위치 성향 (model_12)
        S, N = ac_sn(q, ac[0], ac[1], season_f)
        d = S / (N + ac[2])
        P = shift_mean(P, d[:, 0], d[:, 1])
    (S1, N1), (S2, N2) = loc_sn(q, half_life, season_f)
    M = fc.shrink(S2, N2, fc.shrink(S1, N1, P, k_type), k_hand)  # 투수×구종 -> 투수×구종×손
    if var is not None:  # 분산 따로 (model_13): var = (hl_var, k_var, unc, k0)
        v, n, V0 = own_var(q, year, None if ac is None else ac[0], var[0], season_f, var[3])
        M = apply_var(M, v, n, V0, var[1], var[2], N2, k_hand)
    return M

def loc_given_type(q, year, t, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, var=None, cov=True):  # 구종이 t일 때 24칸 확률 (n, 24)과 좌표 정규분포
    qt = fc.as_type(q, t)
    g = fc.gauss(loc_online(qt, year, half_life, k_type, k_hand, ref, ref_hl, season_f, ac, var), qt)
    return zone_probs(g, qt, cov), g

# 재현: var = None -> model_12
yz_v = q_val["zone"].map(fc.ZONE_IDX).to_numpy()
P0, _ = loc_given_type(q_val, fc.VAL_YEAR, q_val["pitch_type"].to_numpy(), LOC_HL, K_TYPE, K_HAND, LOC_REF, LOC_REF_HL, LOC_SEASON_F, LOC_AC, None)
ll0 = -np.log(np.clip(P0[np.arange(len(q_val)), yz_v], 1e-12, None))
assert np.allclose(ll0, fc.align(m12, fp, fc.VAL_YEAR)["ll_zone_true"].to_numpy()), "var = None인데 model_12과 다름"
print(f"var = None -> 2019 실제 구종 기준 24칸 log loss {ll0.mean():.5f} = model_12 (재현 확인)")

var = None -> 2019 실제 구종 기준 24칸 log loss 2.89006 = model_12 (재현 확인)


### 3-1. 분산 따로 + 위치 파라미터 다시 고르기 (2019)

- 연결된 파라미터를 함께: k_hand × 분산 반감 `hl_var` × `k_var` × unc × V0 당김 `k0` (0-2에서 분산을 따로 하면 k_hand가 커짐)
- 거친 격자 전부 -> 범위 끝이면 넓힘 -> 1등 주변 세밀한 격자 전부 -> 그 뒤에도 범위 끝이면 넓힘
- 평균 쪽 파라미터(위치 반감기, 모든 카운트 δ 반감·k_b)는 1등 분산 값에서 주변 작은 격자로 확인
- 대조군 = model_12, 1등 vs 대조군 경기 단위 부트스트랩

In [15]:
def grid_search(score, grid, by="logloss"):  # grid의 모든 조합을 score로 채점 (지표 낮은 순)
    names_ = list(grid)
    combos = pd.MultiIndex.from_product(grid.values(), names=names_).to_frame(index=False)  # 후보의 모든 조합 (한 행 = 한 조합)
    rows = [{**r, **score(**r)} for r in combos.astype(object).where(combos.notna(), None).to_dict("records")]  # NaN -> None으로 되돌려 넘김
    return pd.DataFrame(rows).sort_values(by).reset_index(drop=True)

def finer(cands, best, n=3):
    # 거친 후보 목록에서 best와 양옆 후보 사이를 각각 로그 간격으로 n개씩 나눈 세밀한 후보 (best 가운데, 2n−1개)
    # best가 범위 끝이면 바깥으로 2배 넓힘, best가 None이면 None 그대로
    if best is None:
        return [None]
    nums = sorted(c for c in cands if c is not None)  # None 뺀 숫자 후보
    i = nums.index(best)
    lo = nums[i - 1] if i > 0 else best / 2  # 왼쪽 이웃 (없으면 절반)
    hi = nums[i + 1] if i < len(nums) - 1 else best * 2  # 오른쪽 이웃 (없으면 2배)
    vals = np.r_[np.geomspace(lo, best, n), np.geomspace(best, hi, n)[1:]]  # lo~best, best~hi를 같은 비율 간격으로
    return sorted({float(f"{v:.3g}") for v in vals})  # 유효숫자 3자리로 반올림, 중복 제거

def best_of(table, grid):  # 표 1등 행의 파라미터 dict (NaN -> None, 숫자는 float)
    r = table.iloc[0]
    return {k: (None if pd.isna(r[k]) else float(r[k])) for k in grid}

def marginal(table, grid, by="logloss"):  # 파라미터마다 값별 가장 좋은 by 지표 (나머지는 그 값에서 가장 좋은 조합) -> 어느 파라미터가 민감한지
    out = {}
    for k in grid:
        s = table.assign(**{k: table[k].astype(object).where(table[k].notna(), "None")}).groupby(k, sort=False)[by].min()
        out[k] = s.reindex([("None" if v is None else v) for v in grid[k] if ("None" if v is None else v) in s.index])
    return out

def game_ci(d, gid, B=1000):  # 경기 단위 부트스트랩 95% 구간 (같은 경기 공은 독립이 아님)
    s = pd.DataFrame({"g": gid, "d": d}).groupby("g")["d"].agg(["sum", "count"])
    i = np.random.default_rng(0).integers(0, len(s), (B, len(s)))
    return np.percentile(s["sum"].to_numpy()[i].sum(1) / s["count"].to_numpy()[i].sum(1), [2.5, 97.5])



REF19 = loc_league(q_val, fc.VAL_YEAR, LOC_REF, LOC_REF_HL)  # 2019 기준점 (고정)
gid_v = q_val["gameID"].to_numpy()
_LSN, _ACSN, _OV = {}, {}, {}  # 2019 lookup 캐시

def ll_v(k_hand, hl_var, k_var, unc, k0=0.0, half_life=LOC_HL, hl_b=LOC_AC[1], k_b=LOC_AC[2]):  # 2019 공별 24칸 log loss (실제 구종), hl_var None = 분산 따로 없음
    if half_life not in _LSN:
        _LSN[half_life] = loc_sn(q_val, half_life, LOC_SEASON_F, cache=False)
    if hl_b not in _ACSN:
        _ACSN[hl_b] = ac_sn(q_val, LOC_AC[0], hl_b, LOC_SEASON_F, cache=False)
    (S1, N1), (S2, N2) = _LSN[half_life]
    S, N = _ACSN[hl_b]
    d = S / (N + k_b)
    M = fc.shrink(S2, N2, fc.shrink(S1, N1, shift_mean(REF19, d[:, 0], d[:, 1]), K_TYPE), k_hand)
    if hl_var is not None:
        if (hl_var, k0) not in _OV:
            _OV[(hl_var, k0)] = own_var(q_val, fc.VAL_YEAR, LOC_AC[0], hl_var, LOC_SEASON_F, k0, cache=False)
        M = apply_var(M, *_OV[(hl_var, k0)], k_var, bool(unc), N2, k_hand)
    Pz = zone_probs(fc.gauss(M, q_val), q_val)
    return -np.log(np.clip(Pz[np.arange(len(q_val)), yz_v], 1e-12, None))

def score_v(k_hand, hl_var, k_var, unc, k0=0.0, half_life=LOC_HL, hl_b=LOC_AC[1], k_b=LOC_AC[2]):
    return {"logloss": ll_v(k_hand, hl_var, k_var, unc, k0, half_life, hl_b, k_b).mean()}

LL16 = ll_v(K_HAND, None, None, None)  # 대조군 = model_12
assert np.allclose(LL16, ll0)
NUM = ["k_hand", "hl_var", "k_var", "k0"]  # 숫자 파라미터 (넓히기·세밀한 격자 대상)
GRID_V = {"k_hand": [19.9, 30.0, 45.0], "hl_var": [100.0, 200.0, 400.0], "k_var": [5.48, 17.3, 54.8], "unc": [False, True], "k0": [10.0, 100.0, 1000.0]}  # k0 = 투수 안 분산 V0 당김 (첫 실행 너클 문제로 추가)
res = grid_search(score_v, GRID_V)  # 거친 격자
tried = {k: list(GRID_V[k]) for k in NUM}

def widen(res, label):  # 1등이 범위 끝인 숫자 파라미터를 바깥으로 넓힘 (나머지는 1등 값, 최대 4번)
    for rnd in range(4):
        b = res.iloc[0]
        add = {k: [b[k] / 2] if b[k] == min(tried[k]) else [b[k] * 2] for k in NUM if b[k] in (min(tried[k]), max(tried[k]))}
        if not add:
            break
        print(f"{label} {rnd + 1}번째 넓히기:", add)
        for k, v in add.items():
            tried[k] += v
            res = pd.concat([res, grid_search(score_v, {**{kk: [b[kk]] for kk in GRID_V}, k: v})])
        res = res.drop_duplicates(list(GRID_V)).sort_values("logloss").reset_index(drop=True)
    return res

res = widen(res, "거친 격자 뒤")
b = res.iloc[0]
grid_f = {**{k: finer(tried[k], float(b[k])) for k in NUM}, "unc": [bool(b["unc"])]}  # 1등 주변 세밀한 후보 (unc는 1등 값)
print("거친 1등", b[list(GRID_V)].to_dict(), "-> 세밀한 후보", grid_f)
res = pd.concat([res, grid_search(score_v, grid_f)]).drop_duplicates(list(GRID_V)).sort_values("logloss").reset_index(drop=True)
for k in NUM:
    tried[k] = sorted(set(tried[k]) | set(grid_f[k]))
res = widen(res, "세밀한 격자 뒤")
b = res.iloc[0]
print(f"2019 24칸 log loss (실제 구종): 대조군 model_12 {LL16.mean():.5f} / 1등 {b.logloss:.5f}")
display(res.head(10).round(5))
for k, s in marginal(res, {**{k: tried[k] for k in NUM}, "unc": [False, True]}).items():
    print(k, s.round(5).to_dict())

# 평균 쪽 파라미터 확인 (1등 분산 값에서 주변 작은 격자)
GRID_MEAN = {"k_hand": [float(b.k_hand)], "hl_var": [float(b.hl_var)], "k_var": [float(b.k_var)], "unc": [bool(b.unc)], "k0": [float(b.k0)],
             "half_life": [29.7, 35.4, 42.1, 50.0], "hl_b": [8.84, 12.5, 17.7], "k_b": [10.0, 17.3, 30.0]}
res_m = grid_search(score_v, GRID_MEAN)
MNUM = ["half_life", "hl_b", "k_b"]
mtried = {k: list(GRID_MEAN[k]) for k in MNUM}
for rnd in range(4):  # 평균 쪽도 1등이 범위 끝이면 그 파라미터만 바깥으로 (위치 반감 35.4가 아래쪽 끝이었음 -> 29.7부터, 사용자 요청)
    bm = res_m.iloc[0]
    add = {k: [round(bm[k] / 2 ** 0.25, 2)] if bm[k] == min(mtried[k]) else [round(bm[k] * 2 ** 0.25, 2)] for k in MNUM if bm[k] in (min(mtried[k]), max(mtried[k]))}
    if not add:
        break
    print(f"평균 쪽 {rnd + 1}번째 넓히기:", add)
    for k, v in add.items():
        mtried[k] += v
        res_m = pd.concat([res_m, grid_search(score_v, {**{kk: [bm[kk]] for kk in GRID_MEAN}, k: v})])
    res_m = res_m.drop_duplicates(list(GRID_MEAN)).sort_values("logloss").reset_index(drop=True)
display(res_m.head(5).round(5))
for k in MNUM:
    print("  ", k, res_m.groupby(k)["logloss"].min().round(5).to_dict())
best = res_m.iloc[0]
print(f"평균 쪽 주변 확인: model_12 값(42.1 · 12.5 · 17.3) {res_m[(res_m.half_life == 42.1) & (res_m.hl_b == 12.5) & (res_m.k_b == 17.3)].logloss.iloc[0]:.5f} / 1등 {best.logloss:.5f}")
d = LL16 - ll_v(float(best.k_hand), float(best.hl_var), float(best.k_var), bool(best.unc), float(best.k0), float(best.half_life), float(best.hl_b), float(best.k_b))
lo, hi = game_ci(d, gid_v)
print(f"1등 vs model_12 이득 ×1000: {d.mean() * 1000:.3f} (95% {lo * 1000:.3f} ~ {hi * 1000:.3f})")

거친 1등 {'k_hand': 30.0, 'hl_var': 200.0, 'k_var': 17.3, 'unc': False, 'k0': 100.0} -> 세밀한 후보 {'k_hand': [19.9, 24.4, 30.0, 36.7, 45.0], 'hl_var': [100.0, 141.0, 200.0, 283.0, 400.0], 'k_var': [5.48, 9.74, 17.3, 30.8, 54.8], 'k0': [10.0, 31.6, 100.0, 316.0, 1000.0], 'unc': [False]}
2019 24칸 log loss (실제 구종): 대조군 model_12 2.89006 / 1등 2.88729


,k_hand,hl_var,k_var,unc,k0,logloss
0,30.0,200.0,9.74,False,316.0,2.88729
1,30.0,283.0,9.74,False,316.0,2.88729
2,30.0,283.0,17.30,False,316.0,2.88730
3,30.0,283.0,9.74,False,1000.0,2.88731
4,30.0,200.0,17.30,False,316.0,2.88732
5,30.0,200.0,9.74,False,1000.0,2.88732
6,36.7,200.0,9.74,False,316.0,2.88733
7,36.7,283.0,9.74,False,316.0,2.88734
8,36.7,283.0,17.30,False,316.0,2.88734
9,30.0,400.0,17.30,False,316.0,2.88735


k_hand {19.9: 2.88784, 24.4: 2.88747, 30.0: 2.88729, 36.7: 2.88733, 45.0: 2.88759}
hl_var {100.0: 2.88777, 141.0: 2.88743, 200.0: 2.88729, 283.0: 2.88729, 400.0: 2.88735}
k_var {5.48: 2.88775, 9.74: 2.88729, 17.3: 2.8873, 30.8: 2.88795, 54.8: 2.88931}
k0 {10.0: 2.88755, 31.6: 2.88748, 100.0: 2.88739, 316.0: 2.88729, 1000.0: 2.88731}
unc {False: 2.88729, True: 2.88784}


,k_hand,hl_var,k_var,unc,k0,half_life,hl_b,k_b,logloss
0,30.0,200.0,9.74,False,316.0,35.4,12.5,17.3,2.88724
1,30.0,200.0,9.74,False,316.0,35.4,17.7,30.0,2.88725
2,30.0,200.0,9.74,False,316.0,29.7,17.7,30.0,2.88729
3,30.0,200.0,9.74,False,316.0,35.4,17.7,17.3,2.88729
4,30.0,200.0,9.74,False,316.0,42.1,12.5,17.3,2.88729


   half_life {29.7: 2.88729, 35.4: 2.88724, 42.1: 2.88729, 50.0: 2.8875}
   hl_b {8.84: 2.88733, 12.5: 2.88724, 17.7: 2.88725}
   k_b {10.0: 2.88768, 17.3: 2.88724, 30.0: 2.88725}
평균 쪽 주변 확인: model_12 값(42.1 · 12.5 · 17.3) 2.88729 / 1등 2.88724
1등 vs model_12 이득 ×1000: 2.811 (95% 0.071 ~ 5.693)


#### 선택: 2019 1등 (k_hand 30, 분산 반감 200 · k_var 9.74 · unc 끔 · V0 당김 k0 316, 위치 반감 35.4, 모든 카운트 12.5 · 17.3 그대로)
- 2019 24칸 log loss (실제 구종): model_12 2.89006 -> 2.88724 (−0.0028, 경기 단위 95% 이득 +0.07 ~ +5.69 ×1000)
- k0 316: 구종×손 자유도 합이 수백 이하면 전체 구종 투수 안 분산 쪽으로 많이 감 (10 · 31.6 · 100은 나쁨, 1000과는 거의 같음)
- 분산 따로를 쓰면 k_hand 19.9 -> 30 (평균은 투수 초구 기록을 덜 따라감), unc는 끄는 게 나음
- 범위 끝: 분산 쪽 파라미터는 모두 안쪽 / 평균 쪽은 위치 반감 35.4가 처음 후보(35.4·42.1·50) 아래쪽 끝이었음 -> 29.7 추가 (사용자 요청): 2.88729로 35.4(2.88724)보다 나쁨 -> 35.4는 안쪽 1등, 다시 고른 값 그대로

In [16]:
LOC_HL, K_HAND = float(best.half_life), float(best.k_hand)
LOC_AC = (LOC_AC[0], float(best.hl_b), float(best.k_b))
LOC_VAR = (float(best.hl_var), float(best.k_var), bool(best.unc), float(best.k0))
print(f"위치: 반감 {LOC_HL}, k_type {K_TYPE}, k_hand {K_HAND}, 시즌 초 배수 {LOC_SEASON_F}, 모든 카운트 {LOC_AC}, 분산 따로 (hl_var, k_var, unc, k0) = {LOC_VAR}")

위치: 반감 35.4, k_type 1000.0, k_hand 30.0, 시즌 초 배수 0.707, 모든 카운트 (None, 12.5, 17.3), 분산 따로 (hl_var, k_var, unc, k0) = (200.0, 9.74, False, 316.0)


### 3-2. 진단: 퍼짐이 맞는지, 어디서 좋아졌나 (model_12 vs model_13, 실제 구종 기준)

- 퍼짐 보정 (4-23과 같은 지표): 표준화 잔차 SD (1이면 맞음), 80% 타원 포함률 (0.80이면 맞음) -> 전체 + 다른 카운트 기록량별
- 이득 = model_12 − model_13 공별 24칸 log loss ×1000 (양수 = 좋아짐), 2019 판단 / 2020 확인
- 구종을 몰라서 잃는 몫 (4-23·4-25): 모델이 기대한 몫 vs 실제로 잃은 몫

In [17]:
LOC16 = dict(half_life=42.1, k_type=1000.0, k_hand=19.9, ref=LOC_REF, ref_hl=LOC_REF_HL, season_f=0.707, ac=(None, 12.5, 17.3), var=None)  # model_12 위치
LOC17 = dict(half_life=LOC_HL, k_type=K_TYPE, k_hand=K_HAND, ref=LOC_REF, ref_hl=LOC_REF_HL, season_f=LOC_SEASON_F, ac=LOC_AC, var=LOC_VAR)
_, NH2 = fc.lookup(fc.prior_table(fp, K2, ["x"]), fp, K2, ["x"])  # 투수×구종×손 전날까지 초구 수 (가중 없음)
rows, cal, lost = [], [], {}
for yr, q, msk in [(fc.VAL_YEAR, q_val, val), (fc.TEST_YEAR, q_test, test)]:
    i, yz, yt = np.arange(len(q)), q["zone"].map(fc.ZONE_IDX).to_numpy(), q["pitch_type"].map(fc.TYPE_IDX).to_numpy()
    nll_ = lambda P: -np.log(np.clip(P[i, yz], 1e-12, None))
    _, n_mo, _ = own_var(q, yr, LOC_AC[0], LOC_VAR[0], LOC_SEASON_F, LOC_VAR[3])  # 다른 카운트 기록량 (반감 hl_var 무게 합)
    grp_mo = pd.cut(n_mo, [0, 1, 20, 60, 1e9], right=False, labels=["0", "1–20", "20–60", "60+"])
    LLs = {}
    for nm, L in [("model_12", LOC16), (MODEL, LOC17)]:
        PZall = np.stack([loc_given_type(q, yr, t, **L)[0] for t in CLASSES], 1)  # 구종 9개 각각 가정 (n, 9, 24)
        Pt = P_TYPE_ALL[msk]
        Pmix = np.einsum("nt,ntz->nz", Pt, PZall)
        LLs[nm] = nll_(PZall[i, yt])
        kl = (Pt * (PZall * (np.log(np.clip(PZall, 1e-12, None)) - np.log(np.clip(Pmix, 1e-12, None))[:, None, :])).sum(2)).sum(1)
        lost[(yr, nm)] = {"실제 구종 ll": LLs[nm].mean(), "실전 ll": nll_(Pmix).mean(), "실제로 잃은 몫": (nll_(Pmix) - LLs[nm]).mean(), "모델이 기대한 몫": kl.mean()}
        mx, mz, vx, vz, c = loc_given_type(q, yr, q["pitch_type"].to_numpy(), **L)[1]  # 실제 구종 정규분포
        dx, dz = q["x"].to_numpy() - mx, q["CROSS_PLATE_Z_CM"].to_numpy() - mz
        d2 = (vz * dx ** 2 - 2 * c * dx * dz + vx * dz ** 2) / (vx * vz - c ** 2)
        cd = pd.DataFrame({"ex": dx / np.sqrt(vx), "ez": dz / np.sqrt(vz), "in80": d2 <= -2 * np.log(0.2), "g": np.asarray(grp_mo).astype(str)})
        for gname, gg in [("전체", cd)] + list(cd.groupby("g")):
            cal.append({"연도": yr, "모델": nm, "다른 카운트 기록": gname, "공": len(gg), "SD ex": gg.ex.std(), "SD ez": gg.ez.std(), "80% 타원": gg.in80.mean()})
    d = pd.DataFrame({"이득": (LLs["model_12"] - LLs[MODEL]) * 1000, "다른 카운트 기록": grp_mo,
                      "초구 기록 수": pd.cut(NH2[fp["GYEAR"].eq(yr).to_numpy(), 0], [0, 1, 31, 101, 301, 1e9], right=False, labels=["0", "1–30", "31–100", "101–300", "301+"]),
                      "구종": q["pitch_type"].to_numpy()})
    for var in ["다른 카운트 기록", "초구 기록 수", "구종"]:
        for v, g in d.groupby(var, observed=True):
            rows.append({"연도": yr, "변수": var, "값": str(v), "공": len(g), "이득 ×1000": g["이득"].mean()})
    rows.append({"연도": yr, "변수": "전체", "값": "", "공": len(d), "이득 ×1000": d["이득"].mean()})
    assert np.allclose(LLs["model_12"], fc.align(m12, fp, yr)["ll_zone_true"].to_numpy())  # model_12 재현
print("퍼짐 보정 (실제 구종)")
display(pd.DataFrame(cal).set_index(["다른 카운트 기록", "모델", "연도"]).unstack("연도").round(3))
print("이득")
display(pd.DataFrame(rows).set_index(["변수", "값", "연도"]).unstack("연도").round(2))
print("구종을 몰라서 잃는 몫")
display(pd.DataFrame(lost).T.round(4))

퍼짐 보정 (실제 구종)


공         SD ex         SD ez        80% 타원       
연도                   2019   2020   2019   2020   2019   2020   2019   2020
다른 카운트 기록 모델                                                              
0         model_12     63     31  0.944  1.232  1.036  1.253  0.825  0.710
          model_13     63     31  0.894  1.137  1.031  1.286  0.857  0.742
1–20      model_12    409    139  1.003  1.795  0.949  1.525  0.839  0.748
          model_13    409    139  1.015  1.062  1.009  1.101  0.790  0.770
20–60     model_12   1237    570  0.971  1.029  1.021  0.983  0.808  0.784
          model_13   1237    570  0.985  1.039  1.032  0.991  0.800  0.795
60+       model_12  11749  11875  0.984  0.993  0.983  1.005  0.817  0.807
          model_13  11749  11875  0.978  0.992  0.994  1.013  0.815  0.806
전체        model_12  13458  12615  0.983  1.008  0.986  1.012  0.817  0.805
          model_13  13458  12615  0.979  0.995  0.998  1.014  0.813  0.805

이득


공          이득 ×1000         
연도                    2019     2020     2019     2020
변수        값                                          
구종        CHUP      1654.0   1689.0     6.71     3.54
          CURV      1190.0    951.0    11.34     3.10
          CUTT        57.0    189.0     1.76    27.56
          FAST      6627.0   6047.0    -0.29     0.20
          FORK       720.0    532.0    -4.31     9.52
          KNUC         NaN     17.0      NaN  3321.90
          SLID      2458.0   2434.0     1.94     1.60
          TWOS       752.0    756.0    17.84    -1.84
다른 카운트 기록 0           63.0     31.0   -26.05   -36.22
          1–20       409.0    139.0     4.00   418.97
          20–60     1237.0    570.0     7.20     3.84
          60+      11749.0  11875.0     2.46     1.69
전체                 13458.0  12615.0     2.81     6.29
초구 기록 수   0           89.0     34.0   -34.94   -27.71
          101–300   5672.0   5826.0    -0.13     1.58
          1–30      1620.0    729.0     7.19    83.89
          301+      2461.0   3671.0    -2.89    -0.01
          31–100    3616.0   2355.0    10.27     4.25

구종을 몰라서 잃는 몫


실제 구종 ll   실전 ll  실제로 잃은 몫  모델이 기대한 몫
2019 model_12    2.8901  2.9480    0.0579     0.0622
     model_13    2.8872  2.9471    0.0598     0.0637
2020 model_12    2.8934  2.9448    0.0514     0.0634
     model_13    2.8871  2.9440    0.0569     0.0631

#### 🔍 3-2 결과: V0 고친 뒤 두 해 모두 좋아짐, 2020 숫자는 너클 17구가 크게 부풀림
- 전체 이득 ×1000 (실제 구종): 2019 +2.81, 2020 +6.29
  - 2020 중 너클 17구가 +3,322 (첫 실행 −3,993과 반대, model_12도 너클 몇 구에서 log loss 27까지 나옴) -> **너클을 빼면 2020 약 +1.8** (2019와 같은 방향·비슷한 크기)
  - 2019 커터 57구: 첫 실행 −153 -> +1.8 (V0 고침으로 해결)
- 구종: 커브 +11.3 / +3.1, 체인지업 +6.7 / +3.5, 투심 +17.8 / −1.8, 직구 −0.3 / +0.2 -> 변화구에서 좋아짐
- 다른 카운트 기록량: 60구 이상(공의 90% 이상) +2.5 / +1.7로 고르게 좋아짐, 0구는 −26 / −36 (공 63 · 31개)
- 퍼짐 보정 (80% 타원): 전체 0.813 / 0.805로 model_12과 같음, 기록 적은 1–20구 2020 0.748 -> 0.770 (좁던 것이 나아짐)
- 구종을 몰라서 잃는 몫: 기대 > 실제 차이는 2019 그대로 (0.064 vs 0.060), 2020은 줄어듦 (0.063 vs 0.057)

## 4. 실전 예측 (예측 구종으로 위치 이어서)

- model_12과 같음, 위치만 3-1에서 고른 값 (분산 따로 포함)

In [18]:
def predict(q, year, Pt):  # 초구 q의 공별 예측·점수
    n, i = len(q), np.arange(len(q))
    yt, yz = q["pitch_type"].map(fc.TYPE_IDX).to_numpy(), q["zone"].map(fc.ZONE_IDX).to_numpy()  # 실제 구종·칸 번호
    nll = lambda p: -np.log(np.clip(p, 1e-12, None))  # 공별 log loss (prob_metrics와 같은 방식으로 자름)

    # ① 구종 확률 (n, 9): 2장 결과를 받음

    # ② 위치: 구종 9개 각각 가정 -> 24칸 확률, 실제 위치의 로그 밀도, 예측 중심 (좌우, 높이)
    PZ, LD, MX, MZ = [], [], [], []
    for t in CLASSES:
        P, g = loc_given_type(q, year, t, LOC_HL, K_TYPE, K_HAND, LOC_REF, LOC_REF_HL, LOC_SEASON_F, LOC_AC, LOC_VAR)
        PZ.append(P); LD.append(fc.log_dens(g, q)); MX.append(g[0]); MZ.append(g[1])
    PZ, LD, MX, MZ = np.stack(PZ, 1), np.stack(LD, 1), np.stack(MX, 1), np.stack(MZ, 1)  # (n, 구종, ...)
    joint = Pt[:, :, None] * PZ  # 구종 확률 × 그 구종일 때 칸 확률 = 구종×칸 결합 확률 (n, 9, 24)
    P_zone = joint.sum(1)  # 실전 24칸 확률 = 구종으로 합침
    # 좌표: 구종 확률로 섞은 정규분포에서 실제 위치의 로그 밀도
    ld_mix = np.log(np.einsum("nt,nt->n", Pt, np.exp(LD - LD.max(1, keepdims=True)))) + LD.max(1)  #  공마다 최댓값 빼기 -> exp -> 섞기 -> log (계산 안전)
    dist_true = np.hypot(q["x"] - MX[i, yt], q["CROSS_PLATE_Z_CM"] - MZ[i, yt])  # 실제 구종 중심과의 거리 cm
    dist_mix = np.hypot(q["x"] - (Pt * MX).sum(1), q["CROSS_PLATE_Z_CM"] - (Pt * MZ).sum(1))  # 구종 확률로 가중한 중심과의 거리 cm

    t_hat = np.array(CLASSES)[Pt.argmax(1)]  # 1등 구종 이름

    # 공별 결과 표 (필수: 키, 구종 확률 9개, 실전 24칸 확률 / 나머지는 선택 열)
    return pd.concat([
        q[["gameID", "pit_no", "pitcher_code", "GYEAR", "pitch_type", "zone"]].reset_index(drop=True),
        pd.DataFrame(Pt, columns=fc.P_TYPE),  # p_type_<구종>
        pd.DataFrame(P_zone, columns=fc.P_ZONE),  # p_zone_<칸>
        pd.DataFrame({"t_hat": t_hat}),  # 1등 구종 이름
        pd.DataFrame({
            "ll_type": nll(Pt[i, yt]),  # 구종 log loss
            "ll_zone": nll(P_zone[i, yz]),  # 24칸 log loss (실전)
            "ll_zone_true": nll(PZ[i, yt, yz]),  # 24칸 log loss (실제 구종)
            "ll_joint": nll(joint[i, yt, yz]),  # 구종×칸 log loss
            "nll_coord": -ld_mix,  # 좌표 NLL (실전)
            "nll_coord_true": -LD[i, yt],  # 좌표 NLL (실제 구종)
            "dist": np.asarray(dist_mix),  # 좌표 거리 cm (실전)
            "dist_true": np.asarray(dist_true),  # 좌표 거리 cm (실제 구종)
        })], axis=1)

In [19]:
# 2019 검증 + 2020 평가 초구 전부 예측
pp = pd.concat([predict(q_val, fc.VAL_YEAR, P_TYPE_ALL[val]), predict(q_test, fc.TEST_YEAR, P_TYPE_ALL[test])], ignore_index=True)
print(pp.shape)
# 확인: 위치와 무관한 열(구종)은 model_12와 같아야 함
chk = pp.merge(m12, on=fc.KEYS, suffixes=("", "_12"))
same = ["ll_type"] + fc.P_TYPE
bad = [c for c in same if not np.allclose(chk[c], chk[c + "_12"], equal_nan=True, atol=1e-6)]
assert len(chk) == len(pp) and not bad, f"model_12와 다른 열: {bad}"
print("구종 = model_12 (재현 확인), 위치만 바뀜")

(26073, 48)
구종 = model_12 (재현 확인), 위치만 바뀜


## 5. 평가

- 채점은 `fc.evaluate`, model_12과 나란히 / 경기 단위 부트스트랩 (model_13 − model_12, 2020)
- 2020은 확인용 -> 2020을 보고 파라미터 다시 고르지 않음

In [20]:
def eval_years(pp_):  # 2019·2020 주요 지표 표
    return pd.DataFrame({"검증 2019": fc.evaluate(pp_, fp, fc.VAL_YEAR), "평가 2020": fc.evaluate(pp_, fp, fc.TEST_YEAR)})

summary = eval_years(pp)
m12_sum = eval_years(m12)
display(pd.concat({MODEL: summary, "model_12": m12_sum, "차이 (model_12 대비)": summary - m12_sum}, axis=1).round(4))

model_13          model_12          차이 (model_12 대비)        
                     검증 2019  평가 2020  검증 2019  평가 2020          검증 2019 평가 2020
구종 logloss            1.0613   1.0561   1.0613   1.0561           0.0000  0.0000
구종 top1               0.5385   0.5332   0.5385   0.5332           0.0000  0.0000
구종 top2               0.8177   0.8218   0.8177   0.8218           0.0000  0.0000
구종 top3               0.9374   0.9412   0.9374   0.9412           0.0000  0.0000
구종 macro F1           0.3226   0.3545   0.3226   0.3545           0.0000  0.0000
24칸 logloss (실전)      2.9471   2.9440   2.9480   2.9448          -0.0009 -0.0008
24칸 top1              0.1306   0.1325   0.1314   0.1321          -0.0009  0.0005
24칸 top3              0.2893   0.2953   0.2893   0.2917           0.0000  0.0036
H/S/C logloss         1.0753   1.0759   1.0751   1.0765           0.0001 -0.0006
H/S/C 정확도             0.3979   0.3915   0.3942   0.3910           0.0037  0.0005
24칸 logloss (실제 구종)   2.8872   2.8871   2.8901   2.8934          -0.0028 -0.0063
구종×칸 logloss          3.9485   3.9432   3.9513   3.9489          -0.0028 -0.0057
좌표 평균거리 cm           32.6377  31.8198  32.6480  31.8177          -0.0103  0.0021
좌표 NLL                9.3124   9.2662   9.3152   9.2657          -0.0028  0.0005

In [21]:
display(fc.bootstrap_diff(fc.pitch_scores(m12, fp), fc.pitch_scores(pp, fp)).round(4))  # 음수 = 이 모델이 좋음 (b − a)

,차이 (b−a),95% 하한,95% 상한,판정
ll_type,0.0000,0.0000,0.0000,차이 불확실
ll_zone,-0.0008,-0.0026,0.0012,차이 불확실
ll_joint,-0.0057,-0.0116,-0.0009,좋아짐
nll_coord,0.0005,-0.0019,0.0032,차이 불확실
dist,0.0021,-0.0072,0.0110,차이 불확실


### 🥇1등🥇
#### 2020 24칸 log loss 실제 구종 2.8934 -> 2.8871 (−0.0063, 너클 17구 빼면 약 −0.0018), 실전 2.9448 -> 2.9440, 구종×칸 좋아짐, 나빠진 점수 없음 -> 채택
- 구종은 model_12과 같음 (재현 확인) -> 실전 변화 = 순수한 위치 효과
- 2019 실제 구종 −0.0028 (95% 이득 +0.07 ~ +5.69 ×1000, 하한이 겨우 0 위)
- 부트스트랩 (2020, model_13 − model_12): 구종×칸 −0.0057 (95% −0.0116 ~ −0.0009) 좋아짐, 24칸 −0.0008 · 좌표 NLL +0.0005 · 거리 +0.002cm 차이 불확실
- 2020 숫자는 너클 17구(노경은, 2020 이전 너클 초구 3구)가 크게 흔듦: 첫 실행 −3.7 / 고친 뒤 +6.3 ×1000 중 너클 몫 −5.4 / +4.5, 나머지는 두 번 다 약 +1.7–1.8
- 첫 실행에서 구종×손 투수 안 분산이 표본 적은 구종(너클 3구)에서 너무 작아 2020이 나빠짐 -> 전체 구종 값 쪽으로 당기는 k0 추가 (사용자 확인 후 고침)
- 퍼짐 보정(80% 타원)은 model_12과 같음, 기록 적은 구간의 좁은 퍼짐은 나아짐
- **채택 -> 다음 비교 출발점** (사용자 결정 2026-11-11)

## 6. 저장

- `results/model_13/` (채택하면 `compare.ipynb` `ADOPTED`에 추가)

In [22]:
fc.save_results(MODEL, pp, {"summary": summary})

저장: results\model_13 ['per_pitch.parquet', 'summary.csv']
